# Adaptive Encoder Analysis

Use this notebook after running `scripts/adaptive_encoder_pipeline.py`. It summarizes the adaptive output-dimension sweep, the sparse reduced encoder, and PySR on encoded invariants `J1...Jn`.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

MPLCONFIGDIR = PROJECT_ROOT / ".matplotlib-cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

CONFIG_PATH = PROJECT_ROOT / "configs" / "adaptive_encoder_rotated_hill.toml"
ADAPTIVE_RUN_DIR = PROJECT_ROOT / "results" / "adaptive_rotatedhill"
STAGE1_DIR = ADAPTIVE_RUN_DIR / "stage1"
SPARSE_RUN_DIR = ADAPTIVE_RUN_DIR / "stage2_sparse"
STAGE1_SUMMARY = STAGE1_DIR / "adaptive_stage1_summary.json"
STAGE2_SUMMARY = SPARSE_RUN_DIR / "adaptive_stage2_sparsify.json"
SPARSE_CHECKPOINT = SPARSE_RUN_DIR / "checkpoint_best.pt"
SYMBOLIC_DIR = ADAPTIVE_RUN_DIR / "stage3_pysr"

print("project:", PROJECT_ROOT)
print("config:", CONFIG_PATH)
print("stage1 summary:", STAGE1_SUMMARY)
print("stage2 summary:", STAGE2_SUMMARY)
print("symbolic dir:", SYMBOLIC_DIR)

In [ ]:
import numpy as np
import torch
from pprint import pprint

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import matplotlib.pyplot as plt

from invariant_generator.adaptive import adaptive_sparsification_run_id
from invariant_generator.config import load_config
from invariant_generator.data import prepare_training_data
from invariant_generator.evaluation import evaluate_model, predict_numpy
from invariant_generator.formulas import encoder_formula_report
from invariant_generator.model import InvariantYieldModel
from invariant_generator.utils import resolve_device

config = load_config(CONFIG_PATH)
device = resolve_device(config.train.device)

## Stage 1: Adaptive n Sweep

In [ ]:
with STAGE1_SUMMARY.open("r", encoding="utf-8") as f:
    stage1 = json.load(f)

runs = stage1.get("runs", [])
metric = stage1.get("metric", "rmse")
selected_n = stage1.get("selected_n")
print("metric:", metric)
print("threshold:", stage1.get("threshold"))
print("patience:", stage1.get("patience"))
print("selected n:", selected_n)
print("selected checkpoint:", stage1.get("selected_checkpoint"))

n_values = np.array([row["n"] for row in runs], dtype=int)
train_values = np.array([row["train_metrics"][metric] for row in runs], dtype=float)
test_values = np.array([row["test_metrics"][metric] for row in runs], dtype=float)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(n_values, train_values, marker="o", label=f"train {metric}")
ax.plot(n_values, test_values, marker="o", label=f"test {metric}")
if selected_n is not None:
    ax.axvline(selected_n, color="black", linestyle="--", linewidth=1, label="selected n")
ax.set_xlabel("encoder output dimension n")
ax.set_ylabel(metric)
ax.set_title("Adaptive encoder sweep")
ax.legend()
fig.tight_layout()
plt.show()

for row in runs:
    print(f"n={row['n']:02d} selected={row['selected']} train_{metric}={row['train_metrics'][metric]:.6g} test_{metric}={row['test_metrics'][metric]:.6g}")

### Stage 1 Training Logs

In [ ]:
PLOT_ALL_STAGE1_HISTORIES = True

def load_history_payload(path):
    path = Path(path)
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

history_rows_by_n = {}
for row in runs:
    history_path = row.get("history_path") or str(Path(row["experiment_dir"]) / "history.json")
    payload = load_history_payload(history_path)
    if payload is None:
        print(f"missing history for n={row['n']}: {history_path}")
        continue
    history = payload.get("history", [])
    if history:
        history_rows_by_n[row["n"]] = history

if not history_rows_by_n:
    print("No Stage 1 histories found. New runs save each path in adaptive_stage1_summary.json.")
else:
    selected_histories = history_rows_by_n if PLOT_ALL_STAGE1_HISTORIES else {selected_n: history_rows_by_n.get(selected_n, [])}
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for n, history in sorted(selected_histories.items()):
        if not history:
            continue
        epochs_h = np.asarray([item["epoch"] for item in history], dtype=float)
        for key, ax, title in [
            ("test_mse", axes[0], "Stage 1 test MSE history"),
            ("test_rmse", axes[1], "Stage 1 test RMSE history"),
        ]:
            if any(key in item for item in history):
                values = np.asarray([item.get(key, np.nan) for item in history], dtype=float)
                ax.plot(epochs_h, values, linewidth=1.4, label=f"n={n}")
                ax.set_title(title)
                ax.set_xlabel("epoch")
                ax.set_ylabel(key)
    for ax in axes:
        if ax.lines:
            ax.set_yscale("log")
            ax.legend()
    fig.tight_layout()
    plt.show()

## Stage 2: Sparse Encoder

In [ ]:
with STAGE2_SUMMARY.open("r", encoding="utf-8") as f:
    stage2 = json.load(f)

print("method:", stage2.get("method"))
print("threshold:", stage2.get("threshold"))
print("max active terms per row:", stage2.get("max_active_terms_per_row", 0))
print("source checkpoint:", stage2.get("source_checkpoint"))
print("sparse checkpoint:", stage2.get("checkpoint"))
print("train metrics:")
pprint(stage2.get("train_metrics"))
print("test metrics:")
pprint(stage2.get("test_metrics"))

source_S = np.asarray(stage2["source_S"], dtype=float)
dense_S = np.asarray(stage2["trained_dense_S"], dtype=float)
final_S = np.asarray(stage2["final_sparse_S"], dtype=float)
mask = np.asarray(stage2["mask"], dtype=int)
active_counts = np.asarray(stage2["active_counts"], dtype=int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, values, title in zip(axes, [source_S, dense_S, final_S], ["Stage 1 S", "S after sparsity training", "Final sparse S"]):
    vmax = max(float(np.max(np.abs(values))), 1e-12)
    image = ax.imshow(values, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("input invariant")
    ax.set_ylabel("encoded invariant")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(np.arange(1, len(active_counts) + 1), active_counts)
ax.set_xlabel("encoded invariant J_i")
ax.set_ylabel("active terms")
ax.set_title("Sparse terms per encoded invariant")
plt.show()

### Stage 2 Training Logs

In [ ]:
def load_stage2_history(path_key, inline_key):
    path = stage2.get(path_key)
    if path:
        payload = load_history_payload(path)
        if payload is not None:
            return payload.get("history", [])
    return stage2.get(inline_key, [])

sparse_history = load_stage2_history("sparse_history_path", "sparse_history")
refit_history = load_stage2_history("refit_history_path", "refit_history")

if not sparse_history and not refit_history:
    print("No Stage 2 histories found. Set [sparsification].save_training_logs = true for separate JSON logs.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for history, label in [(sparse_history, "sparsity training"), (refit_history, "masked refit")]:
        if not history:
            continue
        epochs_h = np.asarray([item["epoch"] for item in history], dtype=float)
        for key, ax in [("loss_total", axes[0]), ("loss_data", axes[1])]:
            if any(key in item for item in history):
                values = np.asarray([item.get(key, np.nan) for item in history], dtype=float)
                ax.plot(epochs_h, values, linewidth=1.5, label=label)
                ax.set_xlabel("epoch")
                ax.set_ylabel(key)
    axes[0].set_title("Stage 2 total objective")
    axes[1].set_title("Stage 2 data loss")
    for ax in axes:
        if ax.lines:
            ax.set_yscale("log")
            ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
formulas = stage2.get("formulas", {}).get("formulas", [])
print("Dense-style final formulas in normalized coordinates:")
for item in formulas:
    print(item["normalized_formula"])

print("\nSparse final formulas in raw invariant coordinates:")
for item in formulas:
    print(item["raw_formula"])
    print("  active:", ", ".join(item["active_terms"]) or "none")

## Prediction and Structure Diagnostics

In [ ]:
sparse_config = load_config(CONFIG_PATH)
sparse_config.encoder.output_dim = final_S.shape[0]
sparse_config.train.run_id = adaptive_sparsification_run_id(sparse_config)
data = prepare_training_data(sparse_config)
model = InvariantYieldModel.from_config(sparse_config).to(device)
checkpoint = torch.load(SPARSE_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"], strict=False)

metrics = evaluate_model(model, data.X_test, data.y_test, device=device, batch_size=8192)
pprint(metrics)
y_pred = predict_numpy(model, data.X_test, device=device, batch_size=8192)
y_true = data.y_test
error = y_pred - y_true

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_true, y_pred, s=18, alpha=0.75)
low = min(float(y_true.min()), float(y_pred.min()))
high = max(float(y_true.max()), float(y_pred.max()))
axes[0].plot([low, high], [low, high], color="black", linewidth=1)
axes[0].set_title("Test predictions vs targets")
axes[0].set_xlabel("target")
axes[0].set_ylabel("prediction")
axes[1].hist(error, bins=30)
axes[1].set_title("Test prediction error")
axes[1].set_xlabel("prediction - target")
axes[1].set_ylabel("count")
fig.tight_layout()
plt.show()

### Learned Structural Tensors `a` and `A`

Inspect the trained structural tensors after Stage 2. The second-order tensor `a` is decomposed into symmetric and skew parts. The fourth-order tensor `A` is shown in Mandel matrix form, with eigenvalues used to check the PSD constraint.

In [ ]:
from invariant_generator.constraints import fourth_order_to_mandel_matrix, psd_eigenvalues


def _to_numpy(tensor):
    return tensor.detach().float().cpu().numpy()


def _show_matrix(ax, matrix, title, *, cmap="coolwarm", symmetric=True):
    values = np.asarray(matrix, dtype=float)
    if symmetric:
        vmax = max(float(np.max(np.abs(values))), 1e-12)
        image = ax.imshow(values, cmap=cmap, vmin=-vmax, vmax=vmax)
    else:
        image = ax.imshow(values, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(values.shape[1]))
    ax.set_yticks(range(values.shape[0]))
    return image

model.eval()
with torch.no_grad():
    if model.invariant_pool.raw_a is not None:
        raw_a = _to_numpy(model.invariant_pool.raw_a)
        sym_a, skew_a = model.invariant_pool.effective_second_order_parts()
        sym_a_np = _to_numpy(sym_a)
        skew_a_np = _to_numpy(skew_a)
        sym_a_eig = np.linalg.eigvalsh(sym_a_np)
        a_summary = {
            "raw_a_l2": float(np.linalg.norm(raw_a)),
            "symmetric_part_l2": float(np.linalg.norm(sym_a_np)),
            "skew_part_l2": float(np.linalg.norm(skew_a_np)),
            "symmetric_part_eigenvalues": sym_a_eig.tolist(),
        }
        print("second-order structural tensor a:")
        pprint(a_summary)

        fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
        for ax, values, title in zip(
            axes,
            [raw_a, sym_a_np, skew_a_np],
            ["raw a", "symmetric part s", "skew part w"],
        ):
            image = _show_matrix(ax, values, title)
            fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(5.5, 3.5))
        ax.bar(np.arange(1, len(sym_a_eig) + 1), sym_a_eig)
        ax.axhline(0.0, color="black", linewidth=1)
        ax.set_title("Eigenvalues of symmetric part of a")
        ax.set_xlabel("index")
        ax.set_ylabel("eigenvalue")
        fig.tight_layout()
        plt.show()
    else:
        print("second-order structural tensor a is not enabled.")

    if model.invariant_pool.raw_A is not None:
        A = model.invariant_pool.effective_fourth_order_tensor()
        mandel_A = fourth_order_to_mandel_matrix(A)
        eigvals_A = psd_eigenvalues(mandel_A)
        mandel_A_np = _to_numpy(mandel_A)
        eigvals_A_np = _to_numpy(eigvals_A)
        raw_A_np = _to_numpy(model.invariant_pool.raw_A)
        A_summary = {
            "raw_A_shape": list(raw_A_np.shape),
            "raw_A_l2": float(np.linalg.norm(raw_A_np)),
            "mandel_A_l2": float(np.linalg.norm(mandel_A_np)),
            "min_mandel_eigenvalue": float(np.min(eigvals_A_np)),
            "max_mandel_eigenvalue": float(np.max(eigvals_A_np)),
            "condition_number_abs": float(
                np.max(np.abs(eigvals_A_np)) / max(np.min(np.abs(eigvals_A_np)), 1e-12)
            ),
            "psd_passed": bool(np.min(eigvals_A_np) >= -config.constraints.A_psd.tolerance),
        }
        print("fourth-order structural tensor A:")
        pprint(A_summary)

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
        if raw_A_np.shape == (6, 6):
            image = _show_matrix(axes[0], raw_A_np, "raw A factor L")
            fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)
        else:
            axes[0].axis("off")
            axes[0].set_title(f"raw A shape {raw_A_np.shape}")
        image = _show_matrix(axes[1], mandel_A_np, "effective A in Mandel basis")
        fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)
        fig.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(6, 3.8))
        ax.bar(np.arange(1, len(eigvals_A_np) + 1), eigvals_A_np)
        ax.axhline(0.0, color="black", linewidth=1)
        ax.set_title("Eigenvalues of effective A in Mandel basis")
        ax.set_xlabel("index")
        ax.set_ylabel("eigenvalue")
        fig.tight_layout()
        plt.show()
    else:
        print("fourth-order structural tensor A is not enabled.")


## Stage 3: PySR on Encoded Invariants

In [ ]:
metrics_path = SYMBOLIC_DIR / "metrics.json"
formulas_path = SYMBOLIC_DIR / "encoded_invariant_formulas.json"
best_equation_path = SYMBOLIC_DIR / "best_equation.txt"

if metrics_path.exists():
    with metrics_path.open("r", encoding="utf-8") as f:
        symbolic_metrics = json.load(f)
    print("PySR metrics:")
    pprint(symbolic_metrics)
else:
    print("No PySR metrics found yet:", metrics_path)

if best_equation_path.exists():
    print("\nBest equation:")
    print(best_equation_path.read_text(encoding="utf-8"))

if formulas_path.exists():
    with formulas_path.open("r", encoding="utf-8") as f:
        symbolic_formulas = json.load(f)
    print("Encoded invariant formulas used by PySR:")
    for item in symbolic_formulas.get("formulas", []):
        print(item["raw_formula"])